# 11 · Gradient-Based Geometry Inversion

Notebook 10 searched over fault *geometry* with a grid plus a 1-D optimizer —
workable for one parameter, but the number of grid evaluations explodes as
$k^{\,\text{params}}$, and finite-difference optimizers get noisy gradients.

This notebook uses GeoDef's **JAX backend**: the entire forward pipeline
(geometry → patch grid → Okada kernel → Green's matrix → regularized slip →
predicted data → misfit) is differentiable, so an optimizer can follow
**exact gradients** through all of it and search several geometry parameters
at once.

Requires the optional JAX extra:

```bash
pip install geodef[jax]
```

Everything else in the course stays on the default NumPy backend — this
notebook is the advanced extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

geodef.backend.set_backend("jax")   # float64 by default

rng = np.random.default_rng(0)

# --- Recurring teaching scenario (identical to notebook 10) ----------------
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, backend = {geodef.backend.get_backend()}")

## Variable projection: linear inside, nonlinear outside

The trick (same as notebook 10, now differentiable end to end): for any trial
geometry $\boldsymbol\theta$, the *slip* is a **linear** problem, solved
exactly by regularized least squares,

$$\mathbf m^*(\boldsymbol\theta) = \left(\mathbf G(\boldsymbol\theta)^T
\mathbf W \mathbf G(\boldsymbol\theta) + \lambda\,\mathbf L^T\mathbf L
\right)^{-1}\mathbf G(\boldsymbol\theta)^T \mathbf W \mathbf d ,$$

so the objective reduces to a function of geometry alone:
$\Phi(\boldsymbol\theta) = \mathbf r^T \mathbf W\, \mathbf r$ with
$\mathbf r = \mathbf d - \mathbf G(\boldsymbol\theta)\,
\mathbf m^*(\boldsymbol\theta)$. JAX differentiates through the whole
expression — Okada kernel, patch grid, and the linear solve included — so
`geodef.geometry_search` hands L-BFGS-B the exact
$\nabla_{\!\boldsymbol\theta}\Phi$.

The geometry vector is
$\boldsymbol\theta = [e_0, n_0, \text{depth}, \text{strike}, \text{dip},
\text{length}, \text{width}]$ in a local Cartesian frame anchored at
`(ref_lat, ref_lon)`; the `free` argument picks which entries to optimize.

The first call JIT-compiles the kernel (tens of seconds); repeated calls with
the same problem shapes — multi-start, exercises below — reuse the
compilation and run in milliseconds per iteration.

In [ ]:
# Single unknown, exactly notebook 10's problem: recover dip from a bad start
theta0 = np.array([0.0, 0.0, 25e3, 315.0, 30.0, 180e3, 90e3])  # start dip 30

res_dip = geodef.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip"],
    bounds={"dip": (5.0, 45.0)},
    n_length=nL, n_width=nW,
    components="dip",
    smoothing="laplacian", smoothing_strength=1.0,
)
sig_dip = np.sqrt(res_dip.theta_cov[0, 0])
print(f"recovered dip: {res_dip.theta[4]:.2f} ± {sig_dip:.2f} deg   (true 15.0)")
print(f"reduced chi^2: {res_dip.reduced_chi2:.3f},  iterations: {res_dip.n_iterations}")

## Several parameters at once

This is where gradients earn their keep. A grid over dip *and* depth at
notebook 10's resolution would need hundreds of full inversions; the
gradient-based search converges in a few dozen iterations, and the
Gauss-Newton covariance gives first-order error bars on both parameters.

In [ ]:
theta0 = np.array([0.0, 0.0, 35e3, 315.0, 30.0, 180e3, 90e3])  # both wrong

res = geodef.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip", "depth"],
    bounds={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=nL, n_width=nW,
    components="dip",
    smoothing="laplacian", smoothing_strength=1.0,
)
sig = np.sqrt(np.diag(res.theta_cov))
print(f"recovered dip:   {res.theta[4]:8.2f} ± {sig[0]:.2f} deg   (true 15.0)")
print(f"recovered depth: {res.theta[2]/1e3:8.2f} ± {sig[1]/1e3:.2f} km    (true 25.0)")
print(f"reduced chi^2:   {res.reduced_chi2:8.3f}")

In [ ]:
# Recovered slip at the optimal geometry, next to the truth
best_fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=res.theta[2], strike=315.0, dip=res.theta[4],
    length=180e3, width=90e3, n_length=nL, n_width=nW,
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True (dip 15, depth 25 km)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, res.slip, ax=ax2, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax,
                 title=f"Recovered (dip {res.theta[4]:.1f}, "
                       f"depth {res.theta[2]/1e3:.1f} km)",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Practical notes

- **Non-convexity.** The misfit surface over geometry can have local minima.
  Because compiled kernels are cached, restarting from a handful of initial
  guesses is cheap — keep notebook 10's coarse-grid instinct as the seeding
  strategy, then let the gradients do the refinement.
- **Pick λ first.** The inner solve holds `smoothing_strength` fixed; choose
  it with `geodef.abic_curve` at a reasonable starting geometry. On the JAX
  backend the ABIC sweep itself runs as one batched computation.
- **Laptops: explore in float32.** `geodef.backend.set_precision("float32")`
  roughly doubles throughput for sweeps and coarse searches; rerun the final
  inversion in float64 (see `docs/backend.md`).
- **Uncertainties are local.** `theta_cov` is the Gauss-Newton (Laplace)
  covariance at the optimum — a curvature estimate, not a full posterior.

## Outlook: Bayesian geometry, and beyond rectangles

The differentiable objective built here is exactly what gradient-based
samplers (HMC/NUTS) need for *full* posteriors on geometry — the natural
upgrade of notebook 10's MCMC outlook. And the same machinery extends to
triangular meshes, where `geodef.gradients.tri_greens` differentiates with
respect to every **vertex** coordinate: parameterize the mesh with a small
builder (e.g. dip varying along strike on a large strike-slip fault) and the
chain rule delivers the gradients. That is a higher-dimensional search and a
project of its own.

## Exercises

1. **Three parameters.** Free `strike` as well, starting 10° off. Does the
   covariance show trade-offs between strike and the other parameters?
2. **Multi-start.** Run the dip+depth search from four corners of the bounds
   box and compare the optima — the repeated calls reuse the compiled kernel.
3. **Sensitivity to λ.** Repeat the search with the smoothing strength 10×
   larger and smaller. How stable is the recovered geometry?